# Expression plots for manuscript figures


In [ ]:
#import besca as bc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse, io
import os
import time
import logging
import pkg_resources
import seaborn as sns
import itertools
import sys

# Import DESeq2
#from pydeseq2.dds import DeseqDataSet
#from pydeseq2.ds import DeseqStats
#from pydeseq2.default_inference import DefaultInference

sc.logging.print_versions()

# for standard processing, set verbosity to minimum
sc.settings.verbosity = 0  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80)
version = '2.4'
start0 = time.time()

In [ ]:
from matplotlib.patches import Patch

In [ ]:
#define standardized filepaths based on above input
root_path = os.getcwd()
#bescapath_full = os.path.dirname(bc.__file__)
#bescapath = os.path.split(bescapath_full)[0]

analysis_name = 'sw_besca2_cellbender'
species='mouse' ## or mouse for now

results_folder = os.path.join(root_path, 'analyzed/',analysis_name)
results_file = os.path.join(results_folder, analysis_name + '.annotated.h5ad')
results_file_raw = os.path.join(results_folder, analysis_name + '.raw.h5ad')
results_folder_out = results_folder+ '/DE/sc_decoupler/'
#results_folder_pseudo = results_folder+ '/DE/Pseudobulk/'
#results_folder_pseudo = results_folder+ '/DE/PseudobulkRaw/'

clusters='leiden'
split_condition='readout_id' #'experiment' is generally a good default ### change to sampleID

In [ ]:
import decoupler as dc
# Import DESeq2
#from pydeseq2.dds import DeseqDataSet
#from pydeseq2.ds import DeseqStats

In [ ]:
adata = sc.read_h5ad(results_file)
adataraw=sc.read_h5ad(results_file_raw)


In [ ]:
# Store raw counts in layers
adataraw.X = np.round(adataraw.X)
adataraw.layers['counts'] = adataraw.X


In [ ]:
adataraw.layers['normalized']=adata.raw.X.copy()

In [ ]:
adataraw=adataraw[adata.obs.index].copy()

In [ ]:
#adataraw=adataraw[adata.obs.index,adata.var.index].copy()

In [ ]:
#adataraw.X=adata.raw.X.copy()

In [ ]:
#adataraw.var=adata.var.copy()
adataraw.obs=adata.obs.copy()

In [ ]:
adataraw.obsm=adata.obsm.copy()
adataraw.obsp=adata.obsp.copy()

In [ ]:
sc.pl.umap(adataraw,color='celltype_merged0')

In [ ]:
sc.pl.umap(adataraw,color='celltype_merged')

In [ ]:
sc.pl.umap(adata,color='celltype_merged')

In [ ]:
direct_mapping = {
    'pericentral hepatocyte':'hepatocyte',
    'periportal hepatocyte':'hepatocyte',
    'endothelial cell of lymphatic vessel' : 'endothelial cell',
    'blood vessel endothelial cell' : 'endothelial cell',
    'endothelial cell of hepatic sinusoid' : 'LSEC',
    'myofibroblast cell' : 'fibroblast',
    'proliferating cell' : 'not specified',
    'mixed' : 'not specified',
    'macrophage' : 'monocyte',
    'CD4-positive, alpha-beta T cell' : 'T cell',
    'lymphocyte of B lineage' : 'B cell'
}

In [ ]:
adata.obs['celltype_pub'] = adata.obs['celltype_merged1'].copy()
adata.obs['celltype_pub'] = adata.obs['celltype_pub'].replace(direct_mapping)


In [ ]:
sc.pl.umap(adata, color='celltype_pub')

In [ ]:
goi=['cisAAV-CMV-GFP-WPRE','AU040320',"Dbp","Gadd45g","Gadd45a","Nfil3","Nr1d1","Nr1d2","Arntl","Tef","Rab30","Tubb2a","Acot1","Ripk2","Egfr","Cebpa","Tm9sf2","Ppara","Slc1a2","Cldn2","Avpr1a","Vnn1",'Sort1',"Lrp1", "Rpsa",'Kdr',"Ncl","Erbb3","Nectin2","Flt4","Plxnb2","Itgb1","Cdh1","Sdc1","Cd74",'Srebf1','Hes1']
goirec= ["Slc22a1", "Slco1b2", "Slc1a2", "Cldn2", "Vnn1", "Slc19a2", "Fcgrt", "Eng", "Nrp1", "Sort1", "Lrp1", "Kdr", "Cd81", "Ncl", "Erbb3", "Nectin2", "Rpsa", "Stab2", "Bsg", "Cldn5", "Cd59a", "Fbp1", "Ifitm3", "Ly6e", "Cdh1", "Pigr", "Spp1", "Sdc1", "Lamp1", "Abhd2"]
goisex=['AU040320','Ppara','Ppard','Gadd45g','Tlcd4','Wrnip1','Srebf1', 'Hes1']

In [ ]:
def getAverageGeneExpression(sdata, myg, sample):
    # Table with average gene expression per condition
    obs = sdata.raw[:,myg].X.toarray()
    obs = pd.DataFrame(obs, columns=myg, index=sdata.obs[sample])
    average_obs = obs.groupby(level=0).mean()

    #print("Average expression per gene and sample")
    #display(average_obs)
    return average_obs

In [ ]:
def plotBoxPerCelltype (adata, cellcolumn, genetoplot, samplecol, allcells=None):
    if allcells==None:
        allcells=list(set(adata.obs[cellcolumn]))
    myavgs={}
    for mycell in allcells:
        tmpvals=list(getAverageGeneExpression(adata[adata.obs[cellcolumn].isin([mycell])], [genetoplot], samplecol)[genetoplot])
        if len(tmpvals)==len(list(set(adata.obs[samplecol]))):
            myavgs[mycell] = tmpvals
            #print(len(myavgs[mycell]))
    mypd=pd.DataFrame.from_dict(myavgs)
    mypd.index=getAverageGeneExpression(adata[adata.obs[cellcolumn].isin([mycell])], [genetoplot], samplecol).index

    ax=sns.boxplot(data = mypd, color="grey")
    sns.stripplot(data=mypd, color="black", ax=ax).set(ylabel='log cp10k' )
    plt.xticks(rotation=90)
    plt.title(genetoplot)

    

In [ ]:
def plotBoxPerCelltTypeGender (adata, cellcolumn, xtracol, genetoplot, samplecol, allcells=None):
    if allcells==None:
        allcells=list(set(adata.obs[cellcolumn]))
    myavgs={}
    for mycell in allcells:
        tmpvals=list(getAverageGeneExpression(adata[adata.obs[cellcolumn].isin([mycell])], [genetoplot], samplecol)[genetoplot])
        if len(tmpvals)==len(list(set(adata.obs[samplecol]))):
            myavgs[mycell] = tmpvals
            #print(len(myavgs[mycell]))
    mypd=pd.DataFrame.from_dict(myavgs)
    mypd.index=getAverageGeneExpression(adata[adata.obs[cellcolumn].isin([mycell])], [genetoplot], samplecol).index

    metatab=adata.obs.loc[:,[samplecol,xtracol]].drop_duplicates().copy()
    metatab.index=metatab[samplecol]
    metatab=metatab.drop(columns=samplecol)

    mypd=pd.concat([mypd, metatab], axis=1)
     # Melt the DataFrame
    df_melted = mypd.melt(id_vars=xtracol, var_name='Columns', value_name='Values')
   
    # Create the boxplot
    sns.boxplot(x='Columns', y='Values', hue=xtracol, data=df_melted, palette={'Male': 'lightblue', 'Female': 'lightcoral'})
    # Overlay the individual data points
    sns.stripplot(x='Columns', y='Values', hue=xtracol, data=df_melted, 
                  color="black", dodge=True, jitter=True, alpha=0.5).set(ylabel='log cp10k' )
    # Remove the duplicate legend
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles[0:2], labels[0:2])
    plt.xticks(rotation=90)
    plt.title(genetoplot)    

In [ ]:
set(adata.obs['sex'])

In [ ]:
untreated=adata[adata.obs['CONDITION']=='Untreated'].copy()

In [ ]:
#cellcolumn='celltype_pub'
#genetoplot=goi[0]
#samplecol='individual_id'

In [ ]:
allcells=['hepatocyte',
 'cholangiocyte',
 'fibroblast',
 'LSEC',
 'endothelial cell',
 'myeloid leukocyte',
 'T cell',
 'natural killer cell',
 'B cell',  'plasmacytoid dendritic cell','B T cell doublet',
'hematopoietic cell', 'not specified']

In [ ]:
aav9feperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PericentrallHepatocyte_FemaleAAV9-CMV-GFP_vs_FemaleUntreated_Results.csv')
aav9maperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PericentrallHepatocyte_MaleAAV9-CMV-GFP_vs_MaleUntreated_Results.csv')
aav2feperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PericentrallHepatocyte_FemaleAAV9-CMV-GFP_vs_FemaleUntreated_Results.csv')
aav2maperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PericentrallHepatocyte_MaleAAV9-CMV-GFP_vs_MaleUntreated_Results.csv')

In [ ]:
aav9feperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PeriportalHepatocyte_FemaleAAV9-CMV-GFP_vs_FemaleUntreated_Results.csv')
aav9maperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PeriportalHepatocyte_MaleAAV9-CMV-GFP_vs_MaleUntreated_Results.csv')
aav2feperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PeriportalHepatocyte_FemaleAAV9-CMV-GFP_vs_FemaleUntreated_Results.csv')
aav2maperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/PeriportalHepatocyte_MaleAAV9-CMV-GFP_vs_MaleUntreated_Results.csv')

In [ ]:
aav9fehepato=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Hepatocyte_FemaleAAV9-CMV-GFP_vs_FemaleUntreated_Results.csv')
aav9mahepato=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Hepatocyte_MaleAAV9-CMV-GFP_vs_MaleUntreated_Results.csv')
aav2fehepato=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Hepatocyte_FemaleAAV9-CMV-GFP_vs_FemaleUntreated_Results.csv')
aav2mahepato=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Hepatocyte_MaleAAV9-CMV-GFP_vs_MaleUntreated_Results.csv')

In [ ]:
#allde={'AAV9_F_PC':aav9feperic,"AAV9_M_PC":aav9maperic,
#      "AAV9_F_HE":aav9fehepato,"AAV9_M_HE":aav9mahepato,
#      "AAV9_F_PP":aav9feperip,  "AAV9_M_PP":aav9maperip, 
#      'AAV2_F_PC':aav2feperic, "AAV2_M_PC": aav2maperic, 
#      "AAV2_F_HE": aav2feperip,  "AAV2_M_HE": aav2maperip,
#      "AAV2_F_PP": aav2fehepato, "AAV2_M_PP": aav2mahepato}

allde={'AAV9_F_PC':aav9feperic,"AAV9_M_PC":aav9maperic,
      "AAV9_F_PP":aav9feperip,  "AAV9_M_PP":aav9maperip, 
      'AAV2_F_PC':aav2feperic, "AAV2_M_PC": aav2maperic, 
      "AAV2_F_PP": aav2fehepato, "AAV2_M_PP": aav2mahepato}

In [ ]:
fctab={}
fdrtab={}
for key, value in allde.items():
    tmp=value.log2FoldChange
    tmp.index=value.GeneName
    fctab[key]=tmp.copy()
    tmp=value.padj
    tmp.index=value.GeneName
    fdrtab[key]=tmp.copy()

In [ ]:
fctab=pd.DataFrame.from_dict(fctab).fillna(0)
fdrtab=pd.DataFrame.from_dict(fdrtab).fillna(1)

In [ ]:
#fctab.loc[fctab.index.isin(aav9de_peric),:]

In [ ]:
# Create the clustermap
def plotClusterMapAllCond (pivot_df_logFC,pivot_df_pvalues,genetoplot,figheight=14):
    
    Genes_list_df_unique=pd.DataFrame([pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[0] ,
                                   pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[1] ,
                                   pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[2] ]).transpose()
    Genes_list_df_unique.index=list(pivot_df_logFC.columns)
    Genes_list_df_unique.columns=['Treatment','Sex','Region']
    
    #sex_colors = ['#FFC0CB','#00FFFF']
    #treatment_colors = ['#FFA500','#008000']
    #region_colors = ['#00FF00', '#800080', '#FFFF00']

    sex_colors = ['#c66874','#67c1ca']
    treatment_colors = ['#687ac8','#c8b866']
    #region_colors = ['#c98367',  '#6ac989','#9c69c8']
    region_colors = ['#9c69c8','#c98367']

    
    sex_lut = dict(zip(Genes_list_df_unique['Sex'].unique(), sex_colors))
    treatment_lut = dict(zip(Genes_list_df_unique['Treatment'].unique(), treatment_colors))
    region_lut = dict(zip(Genes_list_df_unique['Region'].unique(), region_colors))

    # Convert the conditions to a DataFrame of colors
    col_colors  = pd.DataFrame(index=pivot_df_logFC.columns)
    col_colors['Sex'] = Genes_list_df_unique['Sex'].map(sex_lut)
    col_colors['Treatment'] = Genes_list_df_unique['Treatment'].map(treatment_lut)
    col_colors['Region'] = Genes_list_df_unique['Region'].map(region_lut)
    



    legend_elements = []
    for cond, cmap in zip(['Sex', 'Treatment', 'Region'], [sex_colors, treatment_colors, region_colors]):
        legend_elements.append(Patch(facecolor='none', edgecolor='none', label=cond + ':'))
        for label, color in zip(Genes_list_df_unique[cond].unique(), cmap):
            legend_elements.append(Patch(facecolor=color, label=label))

        
    cg = sns.clustermap(pivot_df_logFC, cmap="vlag", figsize=(8, figheight), linewidths=0.75, linecolor= 'black',
                        dendrogram_ratio=(.175, .025), center=0, vmax=3, vmin=-3, 
                        col_cluster=False, 
                        annot=pivot_df_pvalues, fmt='',
                        col_colors =col_colors, xticklabels=False, square=True, cbar_pos=(0.01, 0.55, 0.05, 0.18))
    cg.ax_row_dendrogram.set_visible(False) #suppress row dendrogram
    cg.ax_col_dendrogram.set_visible(False) #suppress row dendrogram
    cg.cax.set_title('Log2 FC', pad=10)
    cond_legend = cg.ax_heatmap.legend(labels=[f'p-value < 0.05'], frameon=False, handles=[plt.Line2D([], [], marker='*', color='black', linestyle='None', lw=2)], bbox_to_anchor=(0.01, 0.575))
    cg.ax_heatmap.add_artist(cond_legend)
    cg.ax_heatmap.legend(handles=legend_elements, bbox_to_anchor=(0, 1.1), ncol=1, frameon=False)

    # Manually set the x-axis title at the top
    cg.ax_heatmap.set_title('Condition', y=1.125)

    # Optionally, if you want to remove the default x-axis label
    cg.ax_heatmap.set_xlabel('')
    cg.ax_heatmap.set_ylabel('')

    cg.ax_heatmap.set_position([cg.ax_heatmap.get_position().x0, cg.ax_heatmap.get_position().y0,
                               cg.ax_heatmap.get_position().width, cg.ax_heatmap.get_position().height])
    cg.ax_col_colors.set_position([cg.ax_col_colors.get_position().x0, cg.ax_col_colors.get_position().y0 + 0.010,
                                  cg.ax_col_colors.get_position().width, cg.ax_col_colors.get_position().height])

    plt.savefig("figures/Heatmap-"+genetoplot+".jpg",dpi=300)
    plt.show()

In [ ]:
aav9de_peric=['Tlcd4','Gadd45a','Slc8b1','Tmem140','Tef','Gadd45g','Mal2','Ccnf','Eepd1','Pnrc1','Irf2bp2','Btg1','Phlda1','Fgfr3',
 'Cdkn1a','Irs2','Nr1d1','Arntl','1810013L24Rik','Id2','Ppp1r3b','Ubc','Nr1d2','Elovl3',
 'Ddc','Tat','Leo1','Tubb2a','Dbp','Banp','G6pc','Ttr','Desi2','Msrb1','Cnppd1','Ell','Pcsk4','Tns2','Ppard','Wrnip1','Slc6a9']
genetoplot='AAV9DE_Peric_fromSpatial_'

pivot_df_logFC=fctab.loc[fctab.index.isin(aav9de_peric),:]
pivot_df_pvalues=fdrtab.loc[fdrtab.index.isin(aav9de_peric),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')


In [ ]:
sns.set(font_scale=1.1)

In [ ]:
pivot_df_logFC.columns

In [ ]:
### Fixed order:
corder=['AAV2_F_PP', 'AAV2_F_PC',  'AAV9_F_PP',  'AAV9_F_PC','AAV2_M_PP', 'AAV2_M_PC',  'AAV9_M_PP',  'AAV9_M_PC']

In [ ]:
#pivot_df_logFC.loc[:,corder]

In [ ]:
plotClusterMapAllCond (pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,13)

In [ ]:
aav2de_peric=['Hes1',
 'Zfp36l2','mt-Rnr1',
 'Id1','Mir22hg',
 'Cirbp','Dbp', 'Junb','Cald1', 'Tef', 'Srebf1','Id2','Zfp467','Nr1d2']
genetoplot='AAV2DE_Perip_fromSpatial_'

pivot_df_logFC=fctab.loc[fctab.index.isin(aav2de_peric),:]
pivot_df_pvalues=fdrtab.loc[fdrtab.index.isin(aav2de_peric),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')
plotClusterMapAllCond (pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,5.5)


In [ ]:
aav9de_peric=['Tlcd4','Gadd45a','Slc8b1','Tmem140','Tef','Gadd45g','Mal2','Ccnf','Eepd1','Pnrc1','Irf2bp2',
              'Btg1','Phlda1','Fgfr3',
 'Cdkn1a','Irs2','Nr1d1','Arntl','1810013L24Rik','Id2','Ppp1r3b','Ubc','Nr1d2','Elovl3',
 'Ddc','Tat','Leo1','Tubb2a','Dbp','Banp','G6pc','Ttr','Desi2','Msrb1','Cnppd1','Ell','Pcsk4',
              'Tns2','Ppard','Wrnip1','Slc6a9']

aav2de_peric=['Hes1',
 'Zfp36l2','mt-Rnr1',
 'Id1','Mir22hg',
 'Cirbp','Dbp', 'Junb','Cald1', 'Tef', 'Srebf1','Id2','Zfp467','Nr1d2']

genetoplot='AAV9DE_PericPeriport_fromSpatial_'

In [ ]:
aavde=list(set(aav9de_peric).union(set(aav2de_peric)))
genetoplot='AAV9DE_PericPeriport_fromSpatial_'

In [ ]:
pivot_df_logFC=fctab.loc[fctab.index.isin(aavde),:]
pivot_df_pvalues=fdrtab.loc[fdrtab.index.isin(aavde),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')
plotClusterMapAllCond (pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot)


In [ ]:
F_AAV2_diff=pd.read_csv('/analysis_alberto_selectedSamples/analyzed/some_results/female_AVV2_different_CentralPortal.csv')
M_AAV2_diff=pd.read_csv('/analysis_alberto_selectedSamples/analyzed/some_results/male_AVV2_different_CentralPortal.csv')

F_AAV9_diff=pd.read_csv('/analysis_alberto_selectedSamples/analyzed/some_results/female_AVV9_different_CentralPortal.csv')
M_AAV9_diff=pd.read_csv('/analysis_alberto_selectedSamples/analyzed/some_results/male_AVV9_different_CentralPortal.csv')

In [ ]:
set(F_AAV2_diff['GeneName']).intersection(set(F_AAV9_diff['GeneName']))

In [ ]:
set(M_AAV2_diff['GeneName']).intersection(set(M_AAV9_diff['GeneName']))

In [ ]:
diffgoi=set(M_AAV2_diff['GeneName']).intersection(set(M_AAV9_diff['GeneName'])).union(set(F_AAV2_diff['GeneName']).intersection(set(F_AAV9_diff['GeneName'])))

In [ ]:
diffgoi

In [ ]:
aavde=list(diffgoi)
genetoplot='DiffGOI_PericPeriport_fromSpatial_'

In [ ]:
pivot_df_logFC=fctab.loc[fctab.index.isin(aavde),:]
pivot_df_pvalues=fdrtab.loc[fdrtab.index.isin(aavde),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')
plotClusterMapAllCond (pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,5)


In [ ]:
aavde=["Elovl3", "Chka", "Irs2", ,"Ppard","Srebf1","Acot1", "Cpt2","Dbp", "Nfil3", "Nr1d1", "Nr1d2","Tef", 
       "Arntl", "Gadd45a", "Gadd45g", "Irf2bp2",  "Rnf125","Ripk2","Id2"]
genetoplot='DiffGOISelected_PericPeriport_fromSpatial_'

In [ ]:
pivot_df_logFC=fctab.loc[fctab.index.isin(aavde),:]
pivot_df_pvalues=fdrtab.loc[fdrtab.index.isin(aavde),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')
plotClusterMapAllCond (pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,9)


### Second part: TFs and pathways

In [ ]:
# Create the clustermap
def plotClusterMapAllCondPathways(pivot_df_logFC,pivot_df_pvalues,genetoplot,figheight=14,
                                  myannotation='pathways',myvals='activity_score'):
    
    Genes_list_df_unique=pd.DataFrame([pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[0] ,
                                   pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[1] ,
                                   pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[2] ]).transpose()
    Genes_list_df_unique.index=list(pivot_df_logFC.columns)
    Genes_list_df_unique.columns=['Treatment','Sex','Region']
    
    #sex_colors = ['#FFC0CB','#00FFFF']
    #treatment_colors = ['#FFA500','#008000']
    #region_colors = ['#00FF00', '#800080', '#FFFF00']

    sex_colors = ['#c66874','#67c1ca']
    treatment_colors = ['#c8b866','#687ac8',]
    #region_colors = ['#c98367',  '#6ac989','#9c69c8']
    region_colors = ['#9c69c8','#c98367']

    
    sex_lut = dict(zip(Genes_list_df_unique['Sex'].unique(), sex_colors))
    treatment_lut = dict(zip(Genes_list_df_unique['Treatment'].unique(), treatment_colors))
    region_lut = dict(zip(Genes_list_df_unique['Region'].unique(), region_colors))

    # Convert the conditions to a DataFrame of colors
    col_colors  = pd.DataFrame(index=pivot_df_logFC.columns)
    col_colors['Sex'] = Genes_list_df_unique['Sex'].map(sex_lut)
    col_colors['Treatment'] = Genes_list_df_unique['Treatment'].map(treatment_lut)
    col_colors['Region'] = Genes_list_df_unique['Region'].map(region_lut)


    legend_elements = []
    for cond, cmap in zip(['Sex', 'Treatment', 'Region'], [sex_colors, treatment_colors, region_colors]):
        legend_elements.append(Patch(facecolor='none', edgecolor='none', label=cond + ':'))
        for label, color in zip(Genes_list_df_unique[cond].unique(), cmap):
            legend_elements.append(Patch(facecolor=color, label=label))

        
     # Create the clustermap
    cg = sns.clustermap(pivot_df_logFC, cmap="vlag", figsize=(12, figheight), linewidths=0.75, linecolor= 'black',
                        dendrogram_ratio=(.175, .025), center=0, annot=pivot_df_pvalues, fmt='',col_cluster=False, 
                        col_colors =col_colors, xticklabels=False, square=True, cbar_pos=(0.005, 0.55, 0.05, 0.18)) #row_cluster=False,  
    cg.ax_row_dendrogram.set_visible(False) #suppress row dendrogram
    cg.ax_col_dendrogram.set_visible(False) #suppress row dendrogram
    cg.cax.set_title(myvals, pad=10)
    cond_legend = cg.ax_heatmap.legend(labels=[f'p-value < 0.05'], frameon=False, handles=[plt.Line2D([], [], marker='*', color='black', linestyle='None', lw=2)], bbox_to_anchor=(0.01, 0.575))
    cg.ax_heatmap.add_artist(cond_legend)
    cg.ax_heatmap.legend(handles=legend_elements, bbox_to_anchor=(0, 1.15), ncol=1, frameon=False)

    # Manually set the x-axis title at the top
    cg.ax_heatmap.set_title('Condition', y=1.125)

    # Optionally, if you want to remove the default x-axis label
    cg.ax_heatmap.set_xlabel('')
    cg.ax_heatmap.set_ylabel('')

    cg.ax_heatmap.set_position([cg.ax_heatmap.get_position().x0, cg.ax_heatmap.get_position().y0,
                               cg.ax_heatmap.get_position().width, cg.ax_heatmap.get_position().height])
    cg.ax_col_colors.set_position([cg.ax_col_colors.get_position().x0, cg.ax_col_colors.get_position().y0 + 0.010,
                                  cg.ax_col_colors.get_position().width, cg.ax_col_colors.get_position().height])
    #plt.savefig('/home/valdeola/Figs_Bettina/Pathways.jpg', dpi=300)
    plt.savefig("figures/Heatmap-"+genetoplot+"-"+myannotation+".jpg",dpi=300) #_nocluster
    plt.show()

In [ ]:
patperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Pathways_PericentrallHepatocyte_all.csv')
patperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Pathways_PeriportalHepatocyte_all.csv')
ppatperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Pvals_Pathways_PericentrallHepatocyte_all.csv')
ppatperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Pvals_Pathways_PeriportalHepatocyte_all.csv')


In [ ]:
patperic['Region']='PC'
patperip['Region']='PP'
ppatperic['Region']='PC'
ppatperip['Region']='PP'

In [ ]:
patperic

In [ ]:
patdf=pd.concat([patperic,patperip])
ppatdf=pd.concat([ppatperic,ppatperip])

patdf=patdf.loc[patdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()

ppatdf=ppatdf.loc[ppatdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()

patdf['Sex']=['F','F','M','M','F','F','M','M']
ppatdf['Sex']=['F','F','M','M','F','F','M','M']
patdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']
ppatdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

patdf.index=list(patdf['Treatment']+"_"+patdf['Sex']+"_"+patdf['Region'])
ppatdf.index=list(ppatdf['Treatment']+"_"+ppatdf['Sex']+"_"+ppatdf['Region'])

patdf=patdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()
ppatdf=ppatdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()

In [ ]:
mypat=['PI3K','Androgen','Hypoxia','Trail','EGFR','TGFb','WNT','VEGF','NFkB','Estrogen','p53','MAPK','JAK-STAT','TNFa']

In [ ]:
pivot_df_logFC=patdf.loc[patdf.index.isin(mypat),:]
pivot_df_pvalues=ppatdf.loc[ppatdf.index.isin(mypat),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')


In [ ]:
pivot_df_logFC

In [ ]:
plotClusterMapAllCondPathways(pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,9,'pathways')

In [ ]:
pivot_df_logFC

#### TFs

In [ ]:
tfperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_TFs_PericentrallHepatocyte_all.csv')
tfperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_TFs_PeriportalHepatocyte_all.csv')
ptfperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Pvals_TFs_PericentrallHepatocyte_all.csv')
ptfperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/Pvals_TFs_PeriportalHepatocyte_all.csv')


In [ ]:
tfperic['Region']='PC'
tfperip['Region']='PP'
ptfperic['Region']='PC'
ptfperip['Region']='PP'

In [ ]:
patdf=pd.concat([tfperic,tfperip])
ppatdf=pd.concat([ptfperic,ptfperip])

patdf=patdf.loc[patdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()

ppatdf=ppatdf.loc[ppatdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()


In [ ]:
patdf.loc[:,'cond']

In [ ]:

patdf['Sex']=['F','F','M','M','F','F','M','M']
ppatdf['Sex']=['F','F','M','M','F','F','M','M']
patdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']
ppatdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

patdf.index=list(patdf['Treatment']+"_"+patdf['Sex']+"_"+patdf['Region'])
ppatdf.index=list(ppatdf['Treatment']+"_"+ppatdf['Sex']+"_"+ppatdf['Region'])

patdf=patdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()
ppatdf=ppatdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()

In [ ]:
mytfs=['Tef','Hif','Nr1d1','Brca1','Smad4','Tp63','Zbtb7a','Hbp1','Ovol1','Bmal1','Clock','Nr6a1','Rela','Smarca4',
       'Zbtb18','Bhlhe41','Myc','Kat2b','Esrrb','Tsc22d1','Tgfb1i1','Neurod6','Pbx1','Pin1','Srebf1','Ppara',
      'Atf6','Nr3c1','Mlxipl','Esrrg','Fhl2','Hnf1a','Nr4a1','Creb3l3','Ppard']

In [ ]:
pivot_df_logFC=patdf.loc[patdf.index.isin(mytfs),:]
pivot_df_pvalues=ppatdf.loc[ppatdf.index.isin(mytfs),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')


In [ ]:
#pivot_df_logFC

In [ ]:
plotClusterMapAllCondPathways(pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,12,'TFs')

### GSEA part - HALLMARK

In [ ]:
#sigperic

In [ ]:
sigperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Signatures_PericentrallHepatocyte_all.csv')
sigperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Signatures_PeriportalHepatocyte_all.csv')


In [ ]:
sigperic_df = sigperic.pivot(index='cond', columns='Term', values='NES')
sigperic_df.reset_index(inplace=True)

psigperic_df = sigperic.pivot(index='cond', columns='Term', values='FDR p-value')
psigperic_df.reset_index(inplace=True)

sigperip_df = sigperip.pivot(index='cond', columns='Term', values='NES')
sigperip_df.reset_index(inplace=True)

psigperip_df = sigperip.pivot(index='cond', columns='Term', values='FDR p-value')
psigperip_df.reset_index(inplace=True)


In [ ]:
sigperic_df['Region']='PC'
sigperip_df['Region']='PP'

psigperic_df['Region']='PC'
psigperip_df['Region']='PP'


In [ ]:
patdf=pd.concat([sigperic_df,sigperip_df])
ppatdf=pd.concat([psigperic_df,psigperip_df])


In [ ]:
patdf

In [ ]:

patdf=patdf.loc[patdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()

ppatdf=ppatdf.loc[ppatdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()


In [ ]:

patdf['Sex']=['F','F','M','M','F','F','M','M']
patdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

patdf.index=list(patdf['Treatment']+"_"+patdf['Sex']+"_"+patdf['Region'])

patdf=patdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()


In [ ]:
ppatdf['Sex']=['F','F','M','M','F','F','M','M']
ppatdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

ppatdf.index=list(ppatdf['Treatment']+"_"+ppatdf['Sex']+"_"+ppatdf['Region'])

ppatdf=ppatdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()


In [ ]:
#ppatdf.columns

In [ ]:
mysigs=[ 'ALLOGRAFT_REJECTION', 'ANDROGEN_RESPONSE',
       'BILE_ACID_METABOLISM', 'CHOLESTEROL_HOMEOSTASIS', 'COAGULATION',
       'COMPLEMENT','E2F_TARGETS',
       'EPITHELIAL_MESENCHYMAL_TRANSITION', 'ESTROGEN_RESPONSE_EARLY',
       'ESTROGEN_RESPONSE_LATE', 'FATTY_ACID_METABOLISM', 'G2M_CHECKPOINT',
       'GLYCOLYSIS',  'HEME_METABOLISM', 'HYPOXIA',
       'IL2_STAT5_SIGNALING', 'IL6_JAK_STAT3_SIGNALING',
       'INFLAMMATORY_RESPONSE', 'INTERFERON_ALPHA_RESPONSE',
       'INTERFERON_GAMMA_RESPONSE', 
        'MTORC1_SIGNALING', 'MYC_TARGETS_V1',
       'MYC_TARGETS_V2', 
       'OXIDATIVE_PHOSPHORYLATION', 'P53_PATHWAY', 
       'PEROXISOME', 
       'TGF_BETA_SIGNALING', 'TNFA_SIGNALING_VIA_NFKB', 'XENOBIOTIC_METABOLISM']

In [ ]:
pivot_df_logFC=patdf.loc[patdf.index.isin(mysigs),:]
pivot_df_pvalues=ppatdf.loc[ppatdf.index.isin(mysigs),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')


In [ ]:
#ppatdf

In [ ]:
#pivot_df_logFC

In [ ]:
#pivot_df_pvalues

In [ ]:
plotClusterMapAllCondPathways(pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,12,'Hallmark','NES')

### GSEA part - KEGG

In [ ]:
#supress row order
def plotClusterMapAllCondPathways(pivot_df_logFC, pivot_df_pvalues, genetoplot, figheight=14,
                                  myannotation='pathways', myvals='activity_score'):

    Genes_list_df_unique = pd.DataFrame([pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[0],
                                         pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[1],
                                         pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[2]]).transpose()
    Genes_list_df_unique.index = list(pivot_df_logFC.columns)
    Genes_list_df_unique.columns = ['Treatment', 'Sex', 'Region']

    sex_colors = ['#c66874', '#67c1ca']
    treatment_colors = ['#c8b866', '#687ac8']
    region_colors = ['#9c69c8', '#c98367']

    sex_lut = dict(zip(Genes_list_df_unique['Sex'].unique(), sex_colors))
    treatment_lut = dict(zip(Genes_list_df_unique['Treatment'].unique(), treatment_colors))
    region_lut = dict(zip(Genes_list_df_unique['Region'].unique(), region_colors))

    col_colors = pd.DataFrame(index=pivot_df_logFC.columns)
    col_colors['Sex'] = Genes_list_df_unique['Sex'].map(sex_lut)
    col_colors['Treatment'] = Genes_list_df_unique['Treatment'].map(treatment_lut)
    col_colors['Region'] = Genes_list_df_unique['Region'].map(region_lut)

    legend_elements = []
    for cond, cmap in zip(['Sex', 'Treatment', 'Region'], [sex_colors, treatment_colors, region_colors]):
        legend_elements.append(Patch(facecolor='none', edgecolor='none', label=cond + ':'))
        for label, color in zip(Genes_list_df_unique[cond].unique(), cmap):
            legend_elements.append(Patch(facecolor=color, label=label))

    # Create the clustermap
    cg = sns.clustermap(pivot_df_logFC, cmap="vlag", figsize=(10,10), linewidths=0.75, linecolor='black',
                        dendrogram_ratio=(.175, .025), center=0, annot=pivot_df_pvalues, fmt='',
                        col_cluster=False, row_cluster=False,  # Disable both row and column clustering
                        col_colors=col_colors, xticklabels=False, square=True, cbar_pos=(0.005, 0.55, 0.03, 0.18)) #(x, y, width, height)

    cg.ax_row_dendrogram.set_visible(False)
    cg.ax_col_dendrogram.set_visible(False)
    cg.cax.set_title(myvals, pad=10)
    cond_legend = cg.ax_heatmap.legend(labels=[f'p-value < 0.05'], frameon=False,
                                       handles=[plt.Line2D([], [], marker='*', color='black', linestyle='None', lw=2)],
                                       bbox_to_anchor=(0.01, 0.575))
    cg.ax_heatmap.add_artist(cond_legend)
    cg.ax_heatmap.legend(handles=legend_elements, bbox_to_anchor=(0, 1.15), ncol=1, frameon=False)

    cg.ax_heatmap.set_title('Condition', y=1.125)
    cg.ax_heatmap.set_xlabel('')
    cg.ax_heatmap.set_ylabel('')

    cg.ax_heatmap.set_position([cg.ax_heatmap.get_position().x0, cg.ax_heatmap.get_position().y0,
                                cg.ax_heatmap.get_position().width, cg.ax_heatmap.get_position().height])
    cg.ax_col_colors.set_position([cg.ax_col_colors.get_position().x0, cg.ax_col_colors.get_position().y0 + 0.010,
                                   cg.ax_col_colors.get_position().width, cg.ax_col_colors.get_position().height])

    plt.savefig("figures/Heatmap-" + genetoplot + "-" + myannotation + ".jpg", dpi=300, pad_inches=0.2)

In [ ]:
sigperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Signatures_KEGG_PericentrallHepatocyte_all.csv')
sigperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Signatures_KEGG_PeriportalHepatocyte_all.csv')


In [ ]:
mykeg=['ALPHA_LINOLEIC_ACID_METABOLISM', 'PANTOTHENATE_AND_COA_BIOSYNTHESIS','ARACHIDONIC_ACID_METABOLISM',
'GLYCEROLIPID_METABOLISM','LYSOSOME','PPAR_SIGNALING_PATHWAY','FATTY_ACID_METABOLISM','PEROXISOME',
'COMPLEMENT_AND_COAGULATION_CASCADES','OTHER_GLYCAN_DEGRADATION','BETA_ALANINE_METABOLISM',
'VALINE_LEUCINE_AND_ISOLEUCINE_DEGRADATION','SPHINGOLIPID_METABOLISM','TGF_BETA_SIGNALING_PATHWAY', 'SPLICEOSOME',
'SYSTEMIC_LUPUS_ERYTHEMATOSUS', 'VIBRIO_CHOLERAE_INFECTION', 'ANTIGEN_PROCESSING_AND_PRESENTATION',
'PROTEIN_EXPORT','STEROID_BIOSYNTHESIS', 'GAP_JUNCTION','PATHOGENIC_ESCHERICHIA_COLI_INFECTION',
'TERPENOID_BACKBONE_BIOSYNTHESIS','RIBOSOME','PROTEASOME','ALZHEIMER_DISEASE','N_GLYCAN_BIOSYNTHESIS',
'TYROSINE_METABOLISM','OXIDATIVE_PHOSPHORYLATION','HUNTINGTONS_DISEASE','PARKINSONS_DISEASE','RENAL_CELL_CARCINOMA',
'VIRAL_MYOCARDITIS','HEMATOPOIETIC_CELL_LINEAGE','OLFACTORY_TRANSDUCTION']

In [ ]:
set(sigperip['Term'])

In [ ]:


sigperic_df = sigperic.pivot(index='cond', columns='Term', values='NES')
sigperic_df.reset_index(inplace=True)

psigperic_df = sigperic.pivot(index='cond', columns='Term', values='FDR p-value')
psigperic_df.reset_index(inplace=True)

sigperip_df = sigperip.pivot(index='cond', columns='Term', values='NES')
sigperip_df.reset_index(inplace=True)

psigperip_df = sigperip.pivot(index='cond', columns='Term', values='FDR p-value')
psigperip_df.reset_index(inplace=True)


sigperic_df['Region']='PC'
sigperip_df['Region']='PP'

psigperic_df['Region']='PC'
psigperip_df['Region']='PP'


patdf=pd.concat([sigperic_df,sigperip_df])
ppatdf=pd.concat([psigperic_df,psigperip_df])


patdf


patdf=patdf.loc[patdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()

ppatdf=ppatdf.loc[ppatdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()



patdf['Sex']=['F','F','M','M','F','F','M','M']
patdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

patdf.index=list(patdf['Treatment']+"_"+patdf['Sex']+"_"+patdf['Region'])

patdf=patdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()


ppatdf['Sex']=['F','F','M','M','F','F','M','M']
ppatdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

ppatdf.index=list(ppatdf['Treatment']+"_"+ppatdf['Sex']+"_"+ppatdf['Region'])

ppatdf=ppatdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()



In [ ]:
sns.set(font_scale=1)

In [ ]:

pivot_df_logFC=patdf.loc[patdf.index.isin(mykeg),:]
pivot_df_pvalues=ppatdf.loc[ppatdf.index.isin(mykeg),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')

# Reorder the columns of pivot_df_logFC and pivot_df_pvalues according to corder and mykeg
pivot_df_logFC = pivot_df_logFC.reindex(index=mykeg, columns=corder)
pivot_df_pvalues = pivot_df_pvalues.reindex(index=mykeg, columns=corder)

In [ ]:
pivot_df_pvalues

In [ ]:

#ppatdf

#pivot_df_logFC

#pivot_df_pvalues
genetoplot='KEGG'
plotClusterMapAllCondPathways(pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,10,'KEGG','NES')

### GSEA part - GO BP

In [ ]:


sigperic=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Signatures_GOBP_PericentrallHepatocyte_all.csv')
sigperip=pd.read_csv('analyzed/sw_besca2_cellbender/DE/sc_decoupler/All_Signatures_GOBP_PeriportalHepatocyte_all.csv')


mykeg=['3_UTR_MEDIATED_MRNA_DESTABILIZATION', 'REGULATION_OF_NUCLEAR_TRANSCRIBED_MRNA_CATABOLIC_PROCESS_DEADENYLATION_DEPENDENT_DECAY', 'HISTONE_H3_ACETYLATION','RHYTHMIC_BEHAVIOR',
       'ORGANIC_ACID_CATABOLIC_PROCESS',   'FATTY_ACID_CATABOLIC_PROCESS','MONOCARBOXYLIC_ACID_CATABOLIC_PROCESS',
       'FATTY_ACID_BETA_OXIDATION', 'LIPID_OXIDATION', 'NEGATIVE_REGULATION_OF_COAGULATION','FIBRINOLYSIS', 
       'PLASMINOGEN_ACTIVATION', 'ELECTRON_TRANSPORT_CHAIN', 'MITOCHONDRIAL_RESPIRATORY_CHAIN_COMPLEX_ASSEMBLY', 
       'NADH_DEHYDROGENASE_COMPLEX_ASSEMBLY', 'MITOCHONDRIAL_ELECTRON_TRANSPORT_NADH_TO_UBIQUINONE', 
       'PROTON_MOTIVE_FORCE_DRIVEN_ATP_SYNTHESIS', 'ATP_SYNTHESIS_COUPLED_ELECTRON_TRANSPORT', 
       'OXIDATIVE_PHOSPHORYLATION', 'RESPIRATORY_ELECTRON_TRANSPORT_CHAIN', 'CYTOPLASMIC_TRANSLATION', 
       'RIBOSOME_BIOGENESIS', 'RRNA_METABOLIC_PROCESS', 'RRNA_PROCESSING', 'NCRNA_PROCESSING', 
       'RIBONUCLEOPROTEIN_COMPLEX_BIOGENESIS','COLLAGEN_BIOSYNTHETIC_PROCESS', 
       'PROTEIN_EXIT_FROM_ENDOPLASMIC_RETICULUM', 'PROTEIN_LOCALIZATION_TO_ENDOPLASMIC_RETICULUM',
       'CHAPERONE_MEDIATED_PROTEIN_FOLDING',  'PROTEIN_FOLDING',  
       'ESTABLISHMENT_OF_PROTEIN_LOCALIZATION_TO_ENDOPLASMIC_RETICULUM',
       'RESPONSE_TO_TOPOLOGICALLY_INCORRECT_PROTEIN','CELLULAR_RESPONSE_TO_TOPOLOGICALLY_INCORRECT_PROTEIN', 
       'CELLULAR_RESPONSE_TO_UNFOLDED_PROTEIN' ]

set(sigperip['Term'])



In [ ]:
sigperic_df = sigperic.pivot(index='cond', columns='Term', values='NES')
sigperic_df.reset_index(inplace=True)

psigperic_df = sigperic.pivot(index='cond', columns='Term', values='FDR p-value')
psigperic_df.reset_index(inplace=True)

sigperip_df = sigperip.pivot(index='cond', columns='Term', values='NES')
sigperip_df.reset_index(inplace=True)

psigperip_df = sigperip.pivot(index='cond', columns='Term', values='FDR p-value')
psigperip_df.reset_index(inplace=True)


sigperic_df['Region']='PC'
sigperip_df['Region']='PP'

psigperic_df['Region']='PC'
psigperip_df['Region']='PP'


patdf=pd.concat([sigperic_df,sigperip_df])
ppatdf=pd.concat([psigperic_df,psigperip_df])


patdf



In [ ]:

patdf=patdf.loc[patdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()

ppatdf=ppatdf.loc[ppatdf['cond'].isin(['FemaleAAV2-CMV-GFP_vs_FemaleUntreated',
                                      'FemaleAAV9-CMV-GFP_vs_FemaleUntreated',
                                      'MaleAAV9-CMV-GFP_vs_MaleUntreated',
                                      'MaleAAV2-CMV-GFP_vs_MaleUntreated']),:].copy()



patdf['Sex']=['F','F','M','M','F','F','M','M']
patdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

patdf.index=list(patdf['Treatment']+"_"+patdf['Sex']+"_"+patdf['Region'])

patdf=patdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()


ppatdf['Sex']=['F','F','M','M','F','F','M','M']
ppatdf['Treatment']=['AAV2','AAV9','AAV2','AAV9','AAV2','AAV9','AAV2','AAV9']

ppatdf.index=list(ppatdf['Treatment']+"_"+ppatdf['Sex']+"_"+ppatdf['Region'])

ppatdf=ppatdf.drop(columns=['cond','Region','Sex','Treatment']).transpose()


In [ ]:
mykeg

In [ ]:
sns.set(font_scale=0.8)


pivot_df_logFC=patdf.loc[patdf.index.isin(mykeg),:]
pivot_df_pvalues=ppatdf.loc[ppatdf.index.isin(mykeg),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')


plotClusterMapAllCondPathways(pivot_df_logFC.loc[:,corder],pivot_df_pvalues.loc[:,corder],genetoplot,10,'GOBP','NES')

In [ ]:
sns.set(font_scale=0.8)


pivot_df_logFC=patdf.loc[patdf.index.isin(mykeg),:]
pivot_df_pvalues=ppatdf.loc[ppatdf.index.isin(mykeg),:]
pivot_df_pvalues = pivot_df_pvalues.applymap(lambda x: '*' if x < 0.05 else ' ')



In [ ]:
pivot_df_logFC=pivot_df_logFC.loc[mykeg,:].copy()
pivot_df_pvalues=pivot_df_pvalues.loc[mykeg,:].copy()

In [ ]:
# Reorder the columns of pivot_df_logFC and pivot_df_pvalues according to corder and mykeg
pivot_df_logFC = pivot_df_logFC.reindex(index=mykeg, columns=corder)
pivot_df_pvalues = pivot_df_pvalues.reindex(index=mykeg, columns=corder)

In [ ]:
pivot_df_logFC = pivot_df_logFC.reindex(index=mykeg, columns=corder)
pivot_df_pvalues = pivot_df_pvalues.reindex(index=mykeg, columns=corder)

In [ ]:
pivot_df_pvalues

In [ ]:
def plotClusterMapAllCondPathways(pivot_df_logFC, pivot_df_pvalues, genetoplot, figheight=14,
                                  myannotation='pathways', myvals='activity_score'):

    Genes_list_df_unique = pd.DataFrame([pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[0],
                                         pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[1],
                                         pd.Series(pivot_df_logFC.columns).str.split('_', expand=True)[2]]).transpose()
    Genes_list_df_unique.index = list(pivot_df_logFC.columns)
    Genes_list_df_unique.columns = ['Treatment', 'Sex', 'Region']

    sex_colors = ['#c66874', '#67c1ca']
    treatment_colors = ['#c8b866', '#687ac8']
    region_colors = ['#9c69c8', '#c98367']

    sex_lut = dict(zip(Genes_list_df_unique['Sex'].unique(), sex_colors))
    treatment_lut = dict(zip(Genes_list_df_unique['Treatment'].unique(), treatment_colors))
    region_lut = dict(zip(Genes_list_df_unique['Region'].unique(), region_colors))

    col_colors = pd.DataFrame(index=pivot_df_logFC.columns)
    col_colors['Sex'] = Genes_list_df_unique['Sex'].map(sex_lut)
    col_colors['Treatment'] = Genes_list_df_unique['Treatment'].map(treatment_lut)
    col_colors['Region'] = Genes_list_df_unique['Region'].map(region_lut)

    legend_elements = []
    for cond, cmap in zip(['Sex', 'Treatment', 'Region'], [sex_colors, treatment_colors, region_colors]):
        legend_elements.append(Patch(facecolor='none', edgecolor='none', label=cond + ':'))
        for label, color in zip(Genes_list_df_unique[cond].unique(), cmap):
            legend_elements.append(Patch(facecolor=color, label=label))

    # Create the clustermap
    cg = sns.clustermap(pivot_df_logFC, cmap="vlag", figsize=(10,10), linewidths=0.75, linecolor='black',
                        dendrogram_ratio=(.175, .025), center=0, annot=pivot_df_pvalues, fmt='',
                        col_cluster=False, row_cluster=False,  # Disable both row and column clustering
                        col_colors=col_colors, xticklabels=False, square=True, cbar_pos=(0.005, 0.55, 0.03, 0.18)) #(x, y, width, height)

    cg.ax_row_dendrogram.set_visible(False)
    cg.ax_col_dendrogram.set_visible(False)
    cg.cax.set_title(myvals, pad=10)
    cond_legend = cg.ax_heatmap.legend(labels=[f'p-value < 0.05'], frameon=False,
                                       handles=[plt.Line2D([], [], marker='*', color='black', linestyle='None', lw=2)],
                                       bbox_to_anchor=(0.01, 0.575))
    cg.ax_heatmap.add_artist(cond_legend)
    cg.ax_heatmap.legend(handles=legend_elements, bbox_to_anchor=(0, 1.15), ncol=1, frameon=False)

    cg.ax_heatmap.set_title('Condition', y=1.125)
    cg.ax_heatmap.set_xlabel('')
    cg.ax_heatmap.set_ylabel('')

    cg.ax_heatmap.set_position([cg.ax_heatmap.get_position().x0, cg.ax_heatmap.get_position().y0,
                                cg.ax_heatmap.get_position().width, cg.ax_heatmap.get_position().height])
    cg.ax_col_colors.set_position([cg.ax_col_colors.get_position().x0, cg.ax_col_colors.get_position().y0 + 0.010,
                                   cg.ax_col_colors.get_position().width, cg.ax_col_colors.get_position().height])

    plt.savefig("figures/Heatmap-" + genetoplot + "-" + myannotation + ".jpg", dpi=300,bbox_inches='tight', pad_inches=0.2)
    plt.show()

In [ ]:
plotClusterMapAllCondPathways(pivot_df_logFC, pivot_df_pvalues, genetoplot, 10, 'GOBP', 'NES')

### Other plots

In [ ]:
sns.set_theme(rc={'figure.figsize':(3,4)},style="whitegrid", palette="pastel")
for genetoplot in goisex:  
    plotBoxPerCelltTypeGender(untreated, 'CONDITION','sex',genetoplot, 'individual_id',['AAV9-CMV-GFP', 'Untreated'])
    plt.savefig("figures/PlotPerCONDITION-perGender-"+genetoplot+".pdf")
    plt.show()

In [ ]:
sns.set_theme(rc={'figure.figsize':(8,4)},style="whitegrid", palette="pastel")
for genetoplot in goisex:  
    plotBoxPerCelltTypeGender(untreated, 'celltype_pub','sex',genetoplot, 'individual_id', allcells)
    plt.savefig("figures/PlotPerCellType-perGender-"+genetoplot+".pdf")
    plt.show()

In [ ]:
set(untreated.obs['celltype_merged0'])

In [ ]:
allcells=['endothelial cell',
 'epithelial cell',
 'fibroblast',
 'hematopoietic cell',
 'hepatocyte']

In [ ]:
sns.set_theme(rc={'figure.figsize':(5,3)},style="whitegrid", palette="pastel")
for genetoplot in goisex:  
    plotBoxPerCelltTypeGender(untreated, 'celltype_merged0','sex',genetoplot, 'individual_id', allcells)
    plt.savefig("figures/PlotPerCellType-perGender-lev0-"+genetoplot+".pdf")
    plt.show()

In [ ]:
sns.set_theme(rc={'figure.figsize':(8,4)},style="whitegrid", palette="pastel")
for genetoplot in goisex:  
    plotBoxPerCelltype (adata, 'celltype_pub',genetoplot, 'individual_id', allcells)
    plt.savefig("figures/PlotPerCellType-"+genetoplot+".pdf")
    plt.show()

In [ ]:
sns.set_theme(rc={'figure.figsize':(8,4)},style="whitegrid", palette="pastel")
for genetoplot in goi:  
    plotBoxPerCelltype (adata, 'celltype_pub',genetoplot, 'individual_id', allcells)
    plt.savefig("figures/PlotPerCellType-"+genetoplot+".pdf")
    plt.show()

In [ ]:
sns.set_theme(rc={'figure.figsize':(8,4)},style="whitegrid", palette="pastel")
for genetoplot in goirec:  
    plotBoxPerCelltype (adata, 'celltype_pub',genetoplot, 'individual_id', allcells)
    plt.savefig("figures/PlotPerCellType-"+genetoplot+".pdf")
    plt.show()

In [ ]:
set(adata.obs['CONDITION'])

In [ ]:
#### This is not used for now but it corresponds to code that can be adapted for showing avg expression per condition
import math

sdata=adata.copy()
myg=goi
tr='CONDITION'
sample='individual_id'

mypalette={"Untreated":"black", "AAV2-CMV-GFP": "red","AAV9-CMV-GFP": "blue"}

def plotBoxplot(sdata, average_obs, myg, sample, tr):
    tmp=sdata.obs[[sample,tr,'sex']].drop_duplicates().merge(average_obs, on=sample, how="left")

    max_cols = 5
    rows = math.ceil(len(myg)/max_cols)
    col = min(max_cols, len(myg))

    fig, axes = plt.subplots(rows, col, figsize=(5*col, 4*rows), gridspec_kw={'wspace': 0.55, 'hspace': 0.4, 'left': 0.25})
    plt.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.1)
    if len(myg) == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, gene in enumerate(myg):
        ax=sns.boxplot(data=tmp, y=gene, x=tr, fliersize=0, ax=axes[i], hue='sex', palette=mypalette)
        sns.stripplot(data=tmp, y=gene, x=tr, dodge=True, color="black", jitter=0.2, 
                              size=5, ax=ax, hue='sex')
        dump = ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()], rotation = 0)
        handles, labels = ax.get_legend_handles_labels()
        subset=int(len(handles)/2) # Only legeng for boxplot (half of the handles), not for the dots
        ax.legend(handles[0:subset], labels[0:subset], fontsize='12', title_fontsize='5')
        
    #plt.savefig(figdir+'GeneExpression-pubgenes-'+mysubset+'.pdf')
    #plt.savefig(figdir+'GeneExpression-pubgenes-'+mysubset+'.svg')


#goi=list(set(goi).intersection(set(average_obs.columns)))
#figdir=os.path.join(root_path, 'analyzed', analysis_name+'/figures/')
#plotBoxplot(cdatak, average_obs, pubg, 'individual_id', 'treatment_id')


In [ ]:
adataraw=adataraw[adataraw.obs['celltype_merged0']!='mixed'].copy()

In [ ]:
sc.pl.umap(adataraw,color='celltype_merged0')

In [ ]:
sc.pl.umap(adataraw,color='celltype_merged0')

#### Select some genes of interest to follow

In [ ]:
### these gois are from the pseudobulk DE analysis
goiorder=['Mgll', 'Adgrf2', 'Nr2f2', 'F3', 'Hspb11', 'Per3', 'Tef', 'Tgfbr2', 
          'Zfp704', 'Ugt2b1', 'Ugt2a3', 'Hhex', 'Phlda1', 'Satb2','Tox','Il15',
          'Xist', 'Cyp2b9', 'Nt5e', 'Ddx3y', 'Kdm5d', 'Eif2s3y', 'Uty', 'Cyp2d9', 'Mup20']

goi=goiorder
#goi=[x.upper() for x in goi]
sc.pl.dotplot(adata,var_names=goi,groupby='celltype_merged0')


In [ ]:
sc.pl.dotplot(adata,var_names=goi,groupby='treatment_id', 
              dot_max=0.6, vmax=2)


#### Create conditions used for pairwise comparisons

In [ ]:

#ax=ups.loc[list(set(goiorder).intersection(set(ups.index))),:].sort_values(by='log2FoldChange')['log2FoldChange'].plot.bar()
#ax.set_ylabel("log2FoldChange")
sc.settings.set_figure_params()


adataraw.obs['MYCOND']=adataraw.obs.sex.astype(str)+adataraw.obs.CONDITION.astype(str)

sc.pl.umap(adataraw, color='MYCOND')


In [ ]:
adata.obs['MYCOND']=adata.obs.sex.astype(str)+adata.obs.CONDITION.astype(str)


In [ ]:
sc.settings.set_figure_params()

In [ ]:
! jupyter nbconvert --to html Expression_plotting.ipynb